## Storage
Container are stateless, anything written inside a container's filesystem disappears when it restarts.
> This would not be the case if Kubernetes was doing a `docker restart`. But Kubernetes actually deletes the old container and creates a brand new one on restart.

The simplest way to have some sort of persistence is to create a volume which can be mounted by containers in a pod allowing them to share data between each other. As an example a backend and logs shipper would mount the same volume where the backend generates log. This volume however is not persistent. It gets destroyed when the pod is terminated (or restarted).

![Simple Volume](./images/simple_volume.png)

A `PersistentVolume` is a storage layer that exists independently of any pod's lifecycle. It is typically a network attached storage. A pod requests a persistent volume through `PersistentVolumeClaim`. This claim specifies how much storage is required and the different access modes. Where does this storage come from? That is defined by `StorageClass` which can represent resources like EBS.

![Simple Volume](./images/persistent_volume.png)

## Storage Configuration
To create a simple volume and associate it with a pod:
```yaml
spec:
  containers:
    - name: app
      image: myapp:v1
      volumeMounts:
        - name: logs-volume
          mountPath: /var/log/app

  volumes:
    - name: logs-volume
      emptyDir: {}
```

The volume starts empty on disk. `kubelet` typically puts it in `/var/lib/kubelet/pods/<pod-uid>/volumes/kubernetes.io~empty-dir/<name>/`.

We can instead choose to place the volume in memory:
```yaml
volumes:
  - name: fast-scratch
    emptyDir:
      medium: Memory
      sizeLimit: 64Mi 
```

We can also choose to combine multiple sources and expose it as one volume:
```yaml
volumes:
  - name: all-in-one
    projected:
      sources:
        - configMap:
            name: my-configmap
        - secret:
            name: my-secret
```

Or specify a path on the node:
```yaml
volumes:
  - name: node-data
    hostPath:
      path: /var/data/myapp     # directory on the node with correct perms
      type: Directory           # fail if it doesn't exist
```

To create a persistent volume, point it to a `persistentVolumeClaim`:
```yaml
volumes:
  - name: data
    persistentVolumeClaim:
      claimName: my-pvc
```

Then we define pvc:
```yaml
apiVersion: v1
kind: PersistentVolumeClaim
metadata:
  name: my-pvc
spec:
  storageClassName: fast-ssd
  accessModes:
    - ReadWriteOnce          
  resources:
    requests:
      storage: 5Gi           
```

Different access modes can be:
- `ReadWriteOnce`: one node can mount it to read-write at a time. Used for most block storage (EBS, PD, etc.).
- `ReadOnlyMany`: read-only by many nodes. Typically for shared static content
- `ReadWriteMany`: read-write by many nodes simultaneously. Examples are NFS, EFS, Azure Files, CephFS
- `ReadWriteOncePod`: read-write by a single pod. This is the strictest single-writer guarantee

An example `StorageClass`:
```yaml
apiVersion: storage.k8s.io/v1
kind: StorageClass
metadata:
  name: fast-ssd
provisioner: ebs.csi.aws.com        # Who creates the storage (CSI driver)
parameters:                         # backend-specific options
  type: gp3
  encrypted: "true"
reclaimPolicy: Delete               # Delete | Retain
volumeBindingMode: WaitForFirstConsumer   # Immediate | WaitForFirstConsumer
allowVolumeExpansion: true           # allow PVC size increases
mountOptions:
  - noatime
```